# ACeDeC Time Series Clustering

This notebook sets up and runs **ACeDeC** (Autoencoder-based Centroid-oriented Deep Cluster) from the [ClustPy](https://github.com/collinleiber/ClustPy) framework on synthetic time series data.

**ACeDeC** is a deep clustering method that:
- Learns a compressed latent representation via an autoencoder
- Simultaneously optimises cluster assignments in that latent space
- Is published as an extension of ENRC (Embedded Non-Redundant Clustering)

---
### Notebook structure
1. Install & import dependencies
2. Generate synthetic time series data (3 distinct patterns)
3. Pre-process data for ACeDeC
4. Run ACeDeC clustering
5. Evaluate & visualise results

## 1. Install & Import Dependencies

In [ ]:
# Run this cell once to install everything you need
import subprocess, sys

packages = [
    "clustpy",        # ACeDeC lives here
    "torch",          # deep learning backend
    "numpy",
    "matplotlib",
    "scikit-learn",   # for evaluation metrics
    "scipy",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from clustpy.deep import ACeDeC

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

import torch
torch.manual_seed(42)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

## 2. Generate Synthetic Time Series Data

We create **4 realistic time series archetypes** — patterns you'd plausibly see in real sensor, ECG, or signal data.  
Noise is set high enough that the clusters are not trivially separable.

| Cluster | Pattern | Real-world analogy |
|---------|---------|-------------------|
| 0 | High-frequency oscillation | Fast vibration / high-rate sensor |
| 1 | Low-frequency oscillation + upward trend | Slow seasonal signal with drift |
| 2 | Cumulative random walk | Stock price / Brownian motion |
| 3 | Damped oscillation | Mechanical resonance decaying over time |

In [ ]:
N_PER_CLASS = 100   # samples per cluster
T = 100             # time steps per series
N_CLUSTERS = 4
NOISE = 0.3         # noise std — high enough to make clusters non-trivial

t = np.linspace(0, 2 * np.pi, T)

def make_high_freq(n, noise=NOISE):
    """Fast oscillation: 4 full cycles across the window."""
    base = np.sin(4 * t)
    return base + np.random.normal(0, noise, (n, T))

def make_trend_sine(n, noise=NOISE):
    """Slow oscillation (1 cycle) with a positive linear drift."""
    base = np.sin(t) + np.linspace(0, 1.5, T)
    return base + np.random.normal(0, noise, (n, T))

def make_random_walk(n, noise=NOISE):
    """Cumulative sum of random steps — Brownian-motion style."""
    steps = np.random.normal(0, 0.15, (n, T))
    return np.cumsum(steps, axis=1) + np.random.normal(0, noise * 0.5, (n, T))

def make_damped(n, noise=NOISE):
    """Sine wave with exponential decay envelope."""
    decay = np.exp(-t / (2 * np.pi))          # decays to ~e^-1 by the end
    base  = np.sin(3 * t) * decay
    return base + np.random.normal(0, noise * 0.7, (n, T))

np.random.seed(42)

X0 = make_high_freq(N_PER_CLASS)
X1 = make_trend_sine(N_PER_CLASS)
X2 = make_random_walk(N_PER_CLASS)
X3 = make_damped(N_PER_CLASS)

X      = np.vstack([X0, X1, X2, X3]).astype(np.float32)   # (400, 100)
y_true = np.repeat(np.arange(N_CLUSTERS), N_PER_CLASS)     # ground truth

print(f"Dataset shape   : {X.shape}  ({N_CLUSTERS} clusters × {N_PER_CLASS} samples × {T} time steps)")
print(f"y_true sample   : {y_true[:5]}  ...  {y_true[-5:]}")

In [ ]:
fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(5 * N_CLUSTERS, 3), sharey=False)

titles  = ["Cluster 0\nHigh-freq oscillation",
           "Cluster 1\nTrend + slow sine",
           "Cluster 2\nRandom walk",
           "Cluster 3\nDamped oscillation"]
colours = ["steelblue", "tomato", "seagreen", "mediumpurple"]
datasets = [X0, X1, X2, X3]

for ax, data, title, col in zip(axes, datasets, titles, colours):
    for i in range(12):                          # show 12 sample traces
        ax.plot(data[i], color=col, alpha=0.35, linewidth=0.8)
    ax.plot(data.mean(axis=0), color="black", linewidth=2, label="mean")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Time step")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Value")
plt.suptitle("Synthetic Time Series — 12 samples + mean per cluster  (noise σ=0.3)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Pre-process

ACeDeC (like most deep clustering methods) works on **flat feature vectors**, so each time series of length T becomes a vector of T values.  
We already have that shape `(N, T)`, so we just standardise.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

print(f"Scaled X — mean ≈ {X_scaled.mean():.4f}, std ≈ {X_scaled.std():.4f}")

## 4. Run ACeDeC

### Key parameters (from `enrc.py` source, line 2252)

| Parameter | Default | What it controls |
|-----------|---------|-----------------|
| `n_clusters` | — | Number of clusters to find |
| `embedding_size` | 20 | Autoencoder bottleneck dimension |
| `pretrain_epochs` | 100 | Epochs for autoencoder-only pre-training |
| `clustering_epochs` | 150 | Epochs for joint clustering + AE optimisation |
| `batch_size` | 128 | Mini-batch size |
| `pretrain_optimizer_params` | `{"lr": 1e-3}` | Dict passed to Adam for pre-training |
| `clustering_optimizer_params` | `{"lr": 1e-4}` | Dict passed to Adam for clustering phase |
| `init` | `"acedec"` | Initialisation strategy (`"acedec"`, `"subkmeans"`, `"random"`, `"sgd"`) |
| `tolerance_threshold` | `None` | Early-stop when NMI(old, new) ≥ 1−threshold |
| `final_reclustering` | `True` | Re-cluster the final embedding after training |
| `random_state` | `None` | Seed for reproducibility |

> **Note:** Learning rates go inside the optimizer param dicts, **not** as a top-level `learning_rate` argument.

In [ ]:
acedec = ACeDeC(
    n_clusters=N_CLUSTERS,                     # 4 clusters
    embedding_size=20,                         # larger bottleneck for T=100 series
    pretrain_epochs=100,                       # AE-only pre-training
    clustering_epochs=150,                     # joint optimisation
    batch_size=64,
    pretrain_optimizer_params={"lr": 1e-3},
    clustering_optimizer_params={"lr": 1e-4},
    init="acedec",
    final_reclustering=True,
    random_state=42,
)

print("Fitting ACeDeC … (this may take 2–5 minutes on CPU)")
acedec.fit(X_scaled)
print("Done!")

y_pred = acedec.labels_
print(f"\nPredicted labels (first 10) : {y_pred[:10]}")
print(f"Unique predicted clusters   : {np.unique(y_pred)}")

## 5. Evaluate & Visualise Results

In [ ]:
ari  = adjusted_rand_score(y_true, y_pred)
nmi  = normalized_mutual_info_score(y_true, y_pred)

print("=" * 40)
print(f"Adjusted Rand Index (ARI) : {ari:.4f}  [1.0 = perfect]")
print(f"Norm. Mutual Info  (NMI)  : {nmi:.4f}  [1.0 = perfect]")
print("=" * 40)

In [ ]:
# --- Confusion-style comparison ---
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Note: cluster labels may be permuted vs ground truth — that is normal.
cm_mat = confusion_matrix(y_true, y_pred)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_mat,
                                 display_labels=[f"Pred {i}" for i in range(N_CLUSTERS)])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_ylabel("True cluster")
ax.set_xlabel("Predicted cluster")
ax.set_title("True vs Predicted cluster assignments")
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualise mean time series per predicted cluster ---
palette = ["steelblue", "tomato", "seagreen", "mediumpurple", "orange"]

fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(5 * N_CLUSTERS, 3), sharey=True)
if N_CLUSTERS == 1:
    axes = [axes]

for k, ax in enumerate(axes):
    mask = y_pred == k
    members = X_scaled[mask]
    for series in members:
        ax.plot(series, color=palette[k % len(palette)], alpha=0.3, linewidth=0.8)
    if members.shape[0] > 0:
        ax.plot(members.mean(axis=0), color="black", linewidth=2, label="mean")
    ax.set_title(f"Predicted cluster {k}  (n={mask.sum()})", fontsize=11)
    ax.set_xlabel("Time step")
    ax.legend()

axes[0].set_ylabel("Scaled value")
plt.suptitle("ACeDeC — Time series per predicted cluster", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2-D PCA of the latent embeddings (coloured by prediction & truth) ---
from sklearn.decomposition import PCA
import torch

# The trained autoencoder is stored as `neural_network_trained_` (from ENRC parent)
ae = acedec.neural_network_trained_
ae.eval()

with torch.no_grad():
    X_tensor = torch.tensor(X_scaled)
    # ClustPy's FeedforwardAutoencoder exposes an encode() method
    Z = ae.encode(X_tensor).cpu().numpy()

print(f"Latent space shape : {Z.shape}")   # should be (N, embedding_size)

pca = PCA(n_components=2, random_state=42)
Z2  = pca.fit_transform(Z)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for label, ax, title in [(y_pred, ax1, "Predicted clusters"), (y_true, ax2, "Ground truth")]:
    scatter = ax.scatter(Z2[:, 0], Z2[:, 1], c=label, cmap="tab10",
                         s=25, alpha=0.8, edgecolors="none")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    plt.colorbar(scatter, ax=ax, label="Cluster")

plt.suptitle("PCA of ACeDeC latent space  (embedding_size=10 → 2-D PCA)", fontsize=13)
plt.tight_layout()
plt.show()

## What to look for

- **ARI / NMI close to 1.0** → ACeDeC separated the 4 archetypes well
- **Confusion matrix** → each true cluster should map cleanly to one predicted cluster (labels may be permuted — that is normal)
- **PCA latent space** → 4 distinct blobs; random walk (C2) is likely the hardest to separate since it has no fixed shape

---

## Things to try next

1. **Increase noise** — set `NOISE = 0.5` or `0.8` and see where ACeDeC starts to struggle  
2. **Harder patterns** — make two clusters both sinusoidal but at slightly different frequencies  
3. **Real data** — UCR Time Series Archive has 100+ labelled datasets: https://www.cs.ucr.edu/~eamonn/time_series_data_2018/  
4. **Compare baselines** — run k-means on the raw flattened series and compare ARI  
5. **Custom autoencoder** — pass a 1-D CNN or LSTM encoder via the `neural_network` parameter instead of the default MLP